In [38]:
import os
import pandas as pd
from tqdm import tqdm

In [45]:
eval_path='data'

# these are the experimentally-validated annotations at t_0 on the target ontology
t0_df = pd.read_csv(os.path.join(eval_path, 't0_known.tsv'), sep='\t')

# list of proteins in *evaluation* set
proteins_to_evaluate = pd.read_csv(os.path.join(eval_path, 'cafa5_targets_toeval.tsv'), header=None)[0].unique()



In [40]:
# Find frequency of each term in the t_minus1 ontology
t0_num_proteins = t0_df['EntryID'].nunique()

naive_frequencies = t0_df.groupby('term')['EntryID'].nunique()/t0_num_proteins

# save frequencies to file. This can be used to generate naive predictions for new proteins
naive_frequencies.to_csv(os.path.join(eval_path, 'cafa5_naive_frequencies.tsv'), sep='\t', header=False)

# Convert the series into a dataframe
df_freq = pd.DataFrame({'term':naive_frequencies.index, 'freq':naive_frequencies.values})

In [41]:
# We only include proteins we will evaluate, otherwise the file will be too large 
# (note: if more proteins are added to evaluation set, the following needs to be rerun, 
#  or run for new proteins in evaluation set and appended to existing file)

In [47]:
# replicate the same score for all proteins we will evaluate
predictions = []
for protein in tqdm(proteins_to_evaluate):
    p_df = df_freq.copy()
    p_df.insert(0, 'EntryID', protein)
    predictions.append(p_df)
naive_pred = pd.concat(predictions)

# a large file (for 25,014 proteins, the file is 30GB)
naive_pred.to_csv(os.path.join(eval_path, 'naive_pred.tsv'), sep='\t', index=False, header=False)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 25014/25014 [00:22<00:00, 1107.79it/s]
